In [ ]:
#pip install geonamescache pandas numpy matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geonamescache

In [2]:
#import danych z pakietu geonamescache

gc = geonamescache.GeonamesCache()
cities = gc.get_cities()

df = pd.DataFrame(
    [
        (v["name"], float(v["latitude"]), float(v["longitude"]), int(v["population"]))
        for v in cities.values()
        if v["countrycode"] == "PL"
    ],
    columns = ["city", "latitude", "longitude", "population"]
).sort_values("population", ascending=False).reset_index(drop=True)

In [17]:
dzielnice_warszawy = ['Mokotów','Praga Południe','Ursynów','Wola', 'Bielany', 'Białołęka', 'Targówek', 'Bemowo', 'Śródmieście', 'Praga Północ',
                      'Ochota', 'Wawer', 'Żoliborz', 'Ursus']

In [18]:
df = df[~df['city'].isin(dzielnice_warszawy)]

In [19]:
df.to_csv('data/df.csv', index=False)

In [20]:
#Utworzenie zestawów miast, na których będzie działać algorytm

small_set_20 = df.head(20)[["city", "latitude", "longitude"]].reset_index(drop=True)
medium_set_50 = df.head(50)[["city", "latitude", "longitude"]].reset_index(drop=True)
large_set_100 = df.head(100)[["city", "latitude", "longitude"]].reset_index(drop=True)

In [4]:
def haversine(lat1, long1, lat2, long2):
    """Obliczanie odległości między miastami na podstawie współrzędnych geograficznych"""
    R = 6371.0 #promień Ziemi

    """Ze względu na kulistość Ziemi, odległość między dwoma punktami liczy się inaczej, niż w przypadku odległości na płaszczyźnie.
    Jednym ze sposobów obliczania takich odległości jest użycie wzoru Haversine."""

    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(long2 - long1)

    """Mała zmiana względem typowego wzoru Haversine - używamy np.sin(dphi/2)**2 zamiast 1-np.cos(dphi) oraz tego semgo zamiennika dla dlambda,
    ponieważ sinus do kwadratu jest w stanie zapewnić większą precyzję, przy liczeniu odległości dla małych a.
    1 - np.cos(dphi) = 2 * np.sin(dphi / 2)**2 -> wyrównuje się w ostatecznym wyniku (return)."""
    
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [9]:
def build_distance_df(df, lat='latitude', long='longitude', name='city'):
    
    """Funkcja buduje symateryczną (n*n) macierz odległości między miastami i zapisuje jako data frame 
    o nazwach kolumn i indeksów odpowiadających nazwom miast."""
    
    n = len(df)
    lats = df[lat].values
    longs = df[long].values

    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i + 1, n):
            d = haversine(lats[i], longs[i], lats[j], longs[j])
            distance_matrix[i,j] = d
            distance_matrix[j,i] = d

    cities = df[name].values

    return pd.DataFrame(distance_matrix, index=cities, columns=cities)

In [21]:
print(build_distance_df(small_set_20).head())

             Warsaw        Łódź      Kraków     Wrocław      Poznań  \
Warsaw     0.000000  117.002433  252.497255  301.728629  278.106577   
Łódź     117.002433    0.000000  192.795117  184.871638  187.651477   
Kraków   252.497255  192.795117    0.000000  235.258451  334.364830   
Wrocław  301.728629  184.871638  235.258451    0.000000  145.497024   
Poznań   278.106577  187.651477  334.364830  145.497024    0.000000   

             Gdańsk    Szczecin   Bydgoszcz      Lublin    Katowice  \
Warsaw   283.446878  453.757693  225.579855  152.739930  259.068009   
Łódź     292.308717  379.958355  180.289301  221.701371  171.020090   
Kraków   485.095753  526.590290  365.598058  227.668881   68.360405   
Wrocław  377.624546  309.076563  234.627334  386.014448  168.812899   
Poznań   244.511744  195.715562  107.731070  408.034096  279.815100   

          Białystok      Gdynia  Częstochowa   Sosnowiec       Radom  \
Warsaw   176.482410  303.030341   206.055548  253.559637   92.454237   
Łó

In [22]:
import os

os.makedirs('data', exist_ok=True)

small_set_20.to_csv('data/cities_20.csv', index=False)
medium_set_50.to_csv('data/cities_50.csv', index=False)
large_set_100.to_csv('data/cities_100.csv', index=False)